# Transformers with Hugging Face

This notebook introduces the **Hugging Face Transformers** library for:
1. **Sentiment analysis** with pre-trained pipelines
2. **Text generation**
3. **Named Entity Recognition**
4. **Zero-shot classification**

In [ ]:
try:
    from transformers import pipeline
    HAS_HF = True
except ImportError:
    HAS_HF = False
    print('transformers not installed -- pip install transformers torch')

import warnings
warnings.filterwarnings('ignore')

## 1. Sentiment Analysis

The `pipeline` API provides a high-level interface. Under the hood, it:
1. Tokenizes text using the model's tokenizer
2. Runs inference through the transformer
3. Post-processes logits into human-readable labels

In [ ]:
if HAS_HF:
    sentiment = pipeline('sentiment-analysis')
    
    texts = [
        "I absolutely loved this course on topological data analysis!",
        "The lecture was confusing and poorly organised.",
        "The food was okay, nothing special.",
        "This breakthrough in persistent homology is remarkable."
    ]
    
    results = sentiment(texts)
    for text, res in zip(texts, results):
        print(f"{res['label']:>8} ({res['score']:.3f})  {text}")

## 2. Text Generation

Auto-regressive models (GPT-2) generate text by predicting the next token given the previous ones:
$$P(x_t | x_1, \ldots, x_{t-1})$$

In [ ]:
if HAS_HF:
    generator = pipeline('text-generation', model='gpt2', max_length=60, num_return_sequences=2)
    
    prompt = "Topological data analysis is a field that"
    outputs = generator(prompt)
    
    print(f"Prompt: {prompt}\n")
    for i, out in enumerate(outputs):
        print(f"Generation {i+1}: {out['generated_text']}\n")

## 3. Named Entity Recognition (NER)

Token classification: assign a label (PERSON, ORG, LOC, etc.) to each token.

In [ ]:
if HAS_HF:
    ner = pipeline('ner', aggregation_strategy='simple')
    
    text = "Professor Leko at AIRINA Labs in Togo published a paper on persistent homology using Python."
    entities = ner(text)
    
    print(f"Text: {text}\n")
    print(f"{'Entity':<25} {'Label':<10} {'Score':<8}")
    print('-' * 45)
    for ent in entities:
        print(f"{ent['word']:<25} {ent['entity_group']:<10} {ent['score']:.3f}")

## 4. Zero-Shot Classification

Classify text into arbitrary categories **without training**, using natural language inference (NLI) models.

In [ ]:
if HAS_HF:
    classifier = pipeline('zero-shot-classification')
    
    text = "The Vietoris-Rips complex provides a multiscale view of point cloud data."
    labels = ['mathematics', 'computer science', 'biology', 'finance', 'literature']
    
    result = classifier(text, candidate_labels=labels)
    print(f"Text: {text}\n")
    for label, score in zip(result['labels'], result['scores']):
        bar = '#' * int(score * 40)
        print(f"{label:<18} {score:.3f} {bar}")

In [ ]:
if HAS_HF:
    # Batch zero-shot classification
    test_texts = [
        "The stock market crashed after the interest rate hike.",
        "CRISPR gene editing shows promise for treating sickle cell disease.",
        "The proof of the Poincare conjecture uses Ricci flow."
    ]
    labels = ['science', 'finance', 'medicine', 'mathematics']
    
    print(f"{'Text (truncated)':<55} {'Prediction':<15} {'Score'}")
    print('-' * 80)
    for text in test_texts:
        result = classifier(text, candidate_labels=labels)
        print(f"{text[:53]:<55} {result['labels'][0]:<15} {result['scores'][0]:.3f}")

## Understanding the Transformer Architecture

The transformer (Vaswani et al., 2017) is built on **self-attention**:
$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Key properties:
- **Parallelisable** (unlike RNNs)
- Captures **long-range dependencies** via attention
- **Pre-training** on large corpora + **fine-tuning** for specific tasks

In [ ]:
if HAS_HF:
    # Inspect tokenization
    from transformers import AutoTokenizer
    
    tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
    text = "Persistent homology computes topological invariants."
    
    tokens = tokenizer.tokenize(text)
    ids = tokenizer.encode(text)
    
    print(f"Text:   {text}")
    print(f"Tokens: {tokens}")
    print(f"IDs:    {ids}")
    print(f"\nNote: BERT uses WordPiece tokenization -- '##' indicates a subword continuation.")

## Key Takeaways

- **Hugging Face pipelines** make state-of-the-art NLP accessible in a few lines.
- **Pre-trained transformers** (BERT, GPT-2, etc.) achieve excellent zero-shot and few-shot performance.
- **Subword tokenization** (BPE, WordPiece) handles rare words and morphology.
- For production: consider model size, latency, and fine-tuning on domain data.

This completes our NLP course notebooks: preprocessing, embeddings, classification, and transformers.